# 01 Data: sessions, gold labels and the forest model

**Question.** Do the three scripted sessions load, does a hand-built forest validate, and do
invalid forests (a cycle, an empty topic) fail with a clear message?

**Status of the data.** The prompts and the gold labels are **drafts** written by Claude Opus 5.5
(`claude-opus-5-5`) on 23 Nov 2026 (`scripts/draft_sessions.py`). The group must review the
prompts for realism and fill in `data/gold/*_template.csv` before any number computed against the
gold labels is reported. Every gold-based number in these notebooks is a draft number.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from forest.io import SESSION_IDS, gold_forest, load_gold, load_sessions
from forest.model import Forest, ForestError, Group, Prompt, Topic

pd.set_option("display.max_colwidth", 120)
sessions = load_sessions()
golds = {sid: load_gold(sid) for sid in SESSION_IDS}

## The sessions
Each prompt has an id, a text, a step and a chat. `access_events` say which earlier prompt the
student re-opened at which step; they make recently used prompts expensive to move (`w[p]`, plan §5.4).

In [2]:
rows = []
for sid, s in sessions.items():
    g = golds[sid]
    rows.append({"session": sid, "prompts": len(s.prompts), "chats": len({p.chat_id for p in s.prompts}),
                 "access events": len(s.access_events), "gold groups": len({x.group for x in g.values()}),
                 "gold trees": len({" > ".join(x.parts[:2]) for x in g.values()}),
                 "gold topics": len({x.path for x in g.values()}), "draft": s.draft, "purpose": s.description})
pd.DataFrame(rows)

,session,prompts,chats,access events,gold groups,gold trees,gold topics,draft,purpose
0,s1_focused,30,2,5,1,3,9,True,"One course (neural networks exercise sheet); many prompts on backpropagation, so topics overflow and splits become w..."
1,s2_mixed,30,2,4,3,9,12,True,"Three subjects (linear algebra, probability, databases) interleaved across two chats, so new trees and new groups ar..."
2,s3_drift,25,2,4,1,4,9,True,"Starts broad (machine learning basics), then narrows into attention and transformers, so renames and re-parenting be..."


In [3]:
s1 = sessions["s1_focused"]
pd.DataFrame([{"step": p.step, "chat": p.chat_id, "text": p.text, "gold path": golds["s1_focused"][p.id].path}
              for p in s1.prompts]).head(12)

,step,chat,text,gold path
0,1,c1,We're doing backprop on exercise sheet 4. Can you explain what backpropagation actually computes? I get that it's th...,Deep Learning > Backpropagation
1,2,c1,For a 2-layer MLP with sigmoid hidden units I wrote dL/dW1 = dL/dy * W2 * x. Is that missing something?,Deep Learning > Backpropagation > Chain rule
2,3,c1,how does the chain rule work on a computational graph when one node feeds into two others?,Deep Learning > Backpropagation > Chain rule
3,4,c1,Why do people use ReLU instead of sigmoid in hidden layers?,Deep Learning > Activation functions
4,5,c1,I computed the gradient of the loss wrt the bias in layer 1 as the sum of the deltas over the batch. Is summing righ...,Deep Learning > Backpropagation
5,6,c1,what is the derivative of tanh? I think it's 1 - tanh^2 but I'm not sure why,Deep Learning > Activation functions
6,7,c1,"In the computational graph for f = (x+y)*z, what are the local gradients at each node?",Deep Learning > Backpropagation > Chain rule
7,8,c1,Our network trains fine with 3 layers but with 10 layers the loss barely moves. Is that the vanishing gradient problem?,Deep Learning > Backpropagation > Vanishing gradients
8,9,c1,What's the derivative of softmax combined with cross-entropy? The sheet says it simplifies to y_hat - y,Deep Learning > Backpropagation > Softmax and cross-entropy gradient
9,10,c1,"Leaky ReLU vs ReLU, does the small negative slope really matter in practice?",Deep Learning > Activation functions


## The gold forests
The forest the draft labels describe after the last prompt, as an outline (`(n)` = prompts attached).

In [4]:
for sid, s in sessions.items():
    f = gold_forest(s, golds[sid], len(s.prompts) + 1).validate(D=3)
    print(f"--- {sid}")
    print(f.render())

--- s1_focused
[Deep Learning]
  - Activation functions (3)
  - Backpropagation (5)
    - Chain rule (4)
    - Gradient checking (2)
    - Softmax and cross-entropy gradient (2)
    - Vanishing gradients (5)
  - Training (0)
    - Optimizers (5)
    - Regularization (2)
    - Weight initialization (2)
--- s2_mixed
[Linear Algebra]
  - Eigenvalues (3)
    - Diagonalization (2)
  - Matrix rank (2)
  - SVD (3)
[Probability]
  - Conditional probability (3)
    - Bayes' theorem (2)
  - Distributions (2)
  - Random variables (1)
    - Expectation and variance (2)
[Databases]
  - Indexes (3)
  - Normalization (3)
  - SQL joins (4)
--- s3_drift
[Machine Learning]
  - ML basics (1)
    - Model evaluation (3)
  - Overfitting (3)
  - Sequence models (3)
  - Transformers (2)
    - Attention (6)
    - Masking (2)
    - Multi-head attention (2)
    - Positional encoding (3)


## A hand-built forest validates and survives a JSON round trip

In [5]:
f = Forest(step=3)
f.groups["g1"] = Group("g1", "Deep Learning")
f.topics["t1"] = Topic("t1", "Backpropagation", "g1", "group")
f.topics["t2"] = Topic("t2", "Vanishing gradients", "t1", "topic")
for i, t in [(1, "t1"), (2, "t1"), (3, "t2")]:
    f.prompts[f"p{i}"] = Prompt(f"p{i}", f"prompt {i}", i, "demo")
    f.assignment[f"p{i}"] = t
    f.access_log[f"p{i}"] = i
f.validate(D=3)
assert Forest.from_json(f.to_json()).to_json() == f.to_json()
print(f.render())
print("valid; JSON round trip identical")

[Deep Learning]
  - Backpropagation (2)
    - Vanishing gradients (1)
valid; JSON round trip identical


## Invalid forests fail with a clear message

In [6]:
def show(label, forest, D=3):
    try:
        forest.validate(D)
        print(f"{label}: valid (unexpected)")
    except ForestError as e:
        print(f"{label}:\n  " + "\n  ".join(e.problems))

cyc = f.copy()
cyc.topics["t1"] = Topic("t1", "Backpropagation", "t2", "topic")
show("cycle", cyc)

empty = f.copy()
empty.topics["t3"] = Topic("t3", "Optimizers", "g1", "group")
empty.groups["g2"] = Group("g2", "Linear Algebra")
show("empty topic and group", empty)

show("depth limit D=1", f, D=1)

cycle:
  topic t1 ('Backpropagation') is on a cycle
  topic t2 ('Vanishing gradients') is on a cycle
  group g1 ('Deep Learning') is empty: no trees
empty topic and group:
  topic t3 ('Optimizers') is empty: no prompts and no subtopics
  group g2 ('Linear Algebra') is empty: no trees
depth limit D=1:
  topic t2 ('Vanishing gradients') is at depth 1, deeper than D-1 = 0


## Findings

- All three sessions load: 30, 30 and 25 prompts in 2 chats each, with 4 to 5 access events.
  The draft gold labels give 1 to 3 groups and 9 to 12 topics per session (table above).
- The gold forests validate with `D = 3`. `s1_focused` puts 18 of 30 prompts into the
  Backpropagation tree, so a topic limit of `U = 6` forces splits there, as intended.
- A hand-built forest validates, and its JSON round trip is identical. The same JSON is the
  per-step snapshot format in `outputs/forests/`.
- A cycle, an empty topic, an empty group and a too-deep topic each fail with a message that
  names the topic or group.
- *For the group:* review the prompts for realism, then fill in the gold templates.